# SOME TITLE
---

## Introduction

  # Heading 1
  ## Heading 2
  ### Heading 3

    **Bold text**
  __Bold text__

    *Italic text*
  _Italic text_

    * Item 1
  * Item 2
    - Nested item
   
  1. First item
  2. Second item

- provide some relevant background information on the topic so that someone unfamiliar with it will be prepared to understand the rest of your report
- clearly state the question you tried to answer with your project
- identify and fully describe the dataset that was used to answer the question

## Methods and results

- You may include references if necessary, as long as they all have a consistent citation style.

- describe the methods you used to perform your analysis from beginning to end that narrates the analysis code.

your report should include code which:
- loads data 
- wrangles and cleans the data to the format necessary for the planned analysis
- performs a summary of the data set that is relevant for exploratory data analysis related to the planned analysis 
- creates a visualization of the dataset that is relevant for exploratory data analysis related to the planned analysis
- performs the data analysis
- creates a visualization of the analysis 
- note: all figures should have a figure number and a legend


In [12]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import make_column_transformer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline

In [13]:
import pandas as pd
players = pd.read_csv("https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz")
players

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN
...,...,...,...,...,...,...,...,...,...
191,Amateur,True,b6e9e593b9ec51c5e335457341c324c34a2239531e1890...,0.0,Bailey,Female,17,NaN,NaN
192,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,0.3,Pascal,Male,22,NaN,NaN
193,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,0.0,Dylan,Prefer not to say,17,NaN,NaN
194,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,2.3,Harlow,Male,17,NaN,NaN


In [14]:
'''
1) import, data cleansing, and merge

2) Correlation analysis for variables in players

3) Split-testing

4) K-variables

5) Clustering
'''

'\n1) import, data cleansing, and merge\n\n2) Correlation analysis for variables in players\n\n3) Split-testing\n\n4) K-variables\n\n5) Clustering\n'

In [15]:
### Run this cell before continuing.

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

# Merge through shared hashedEmail
players = pd.read_csv("players.csv")
sessions = pd.read_csv("sessions.csv")
players_merged = players.merge(sessions, on="hashedEmail", how="inner")
players_clean = players_merged.drop(columns=["individualId", "organizationName"])

# Calculate each playtime duration in minutes
players_clean['start_time'] = pd.to_datetime(players_clean['start_time'], format='%d/%m/%Y %H:%M')
players_clean['end_time'] = pd.to_datetime(players_clean['end_time'], format='%d/%m/%Y %H:%M')
players_clean['duration_minutes'] = (players_clean['end_time'] - players_clean['start_time']).dt.total_seconds() / 60

# Calculate total playtime
agg = (
    players_clean.groupby("hashedEmail").agg(
        total_sessions=("hashedEmail", "count"),
        avg_session_length_minutes=("duration_minutes", "mean"),
        total_session_time_minutes=("duration_minutes", "sum")
    ).reset_index()
)

# Merge agg with players
players_total = players_clean.merge(agg, on="hashedEmail", how="inner")

# Get rid of duplicates and unneccessary data
players_total = players_total.drop(columns=["original_start_time", "original_end_time", "start_time", "end_time", "duration_minutes", "played_hours"])
players_total = players_total.drop_duplicates()



# Check to see if we can do clustering (Nope)
columns_to_plot = players_total.loc[:, "age" : "total_session_time_minutes"].columns.tolist()

pm_pairs = alt.Chart(players_total).mark_circle(opacity=0.2).encode(
    alt.X(alt.repeat("row"), type="quantitative"),
    alt.Y(alt.repeat("column"), type="quantitative"),
).properties(
    width=150,
    height=150
).repeat(
    column=columns_to_plot,
    row=columns_to_plot
)
pm_pairs

# Add new categorical column based on a threshold

# use 75th percentile as threshold
threshold = players_total["total_session_time_minutes"].quantile(0.75)

players_total["high_data_player"] = (
    (players_total["total_session_time_minutes"] >= threshold) &
    (players_total["subscribe"] == True)
)

players_total



,experience,subscribe,hashedEmail,name,gender,age,total_sessions,avg_session_length_minutes,total_session_time_minutes,high_data_player
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,Morgan,Male,9,27,74.777778,2019.0,True
27,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,Christian,Male,17,3,85.000000,255.0,True
30,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,Blake,Male,17,1,5.000000,5.0,False
31,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,Flora,Female,21,1,50.000000,50.0,False
32,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,Kylie,Male,21,1,9.000000,9.0,False
...,...,...,...,...,...,...,...,...,...,...
1525,Veteran,True,ba24bebe588a34ac546f8559850c65bc90cd9d51b82158...,Gabriela,Female,44,1,11.000000,11.0,False
1526,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,Pascal,Male,22,1,21.000000,21.0,False
1527,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,Dylan,Prefer not to say,17,1,5.000000,5.0,False
1528,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,Harlow,Male,17,6,29.833333,179.0,False


In [16]:
pm_pairs

alt.RepeatChart(...)

In [17]:
# select only the numeric columns for corr
num_cols = columns_to_plot

# compute correlation table
corr_table = players_total[num_cols].corr(method="pearson")

corr_table

,age,total_sessions,avg_session_length_minutes,total_session_time_minutes
age,1.000000,-0.061144,-0.033657,-0.066540
total_sessions,-0.061144,1.000000,0.169681,0.790637
avg_session_length_minutes,-0.033657,0.169681,1.000000,0.372092
total_session_time_minutes,-0.066540,0.790637,0.372092,1.000000


In [18]:
tp_train, tp_test = train_test_split(players_total, test_size = 0.25, random_state = 123)
tp_train

,experience,subscribe,hashedEmail,name,gender,age,total_sessions,avg_session_length_minutes,total_session_time_minutes,high_data_player
93,Veteran,True,5a340c0e3d1aa3e579efc625bd3e5bca7fc25f7115b68e...,Zoe,Male,20,2,16.5,33.0,False
39,Amateur,True,3caa832978e0596779f4ee7c686c4592fb6de893925025...,Thatcher,Male,22,1,12.0,12.0,False
1412,Regular,True,d43af3bed5e9f1f31077233697c18f3f988a217bd0376a...,Xia,Female,20,1,32.0,32.0,False
846,Veteran,True,e44041459da2102dc20147ed6f0db4753547be66fc4dde...,Gianna,Male,18,1,26.0,26.0,False
243,Regular,True,f2826fb8dbce4d450348f99cb27ade184b713998d96797...,Zane,Male,10,7,38.0,266.0,True
...,...,...,...,...,...,...,...,...,...,...
1413,Regular,True,c121e4d197469bea90e21c0495001f4e21824adb98cbc6...,Rupert,Male,21,1,9.0,9.0,False
1262,Regular,True,7d71c49cbbce8dcf0276b2bfecfa2d16f22cb31a402455...,Devin,Two-Spirited,99,1,8.0,8.0,False
1255,Amateur,True,2cfed571797b66cc810c32562fc5b0f70b5bec0f525079...,Milo,Male,16,1,6.0,6.0,False
830,Beginner,True,96e190b0bf3923cd8d349eee467c09d1130af143335779...,Ibrahim,Prefer not to say,27,8,21.0,168.0,True


In [19]:
tp_preprocessor = make_column_transformer(
    (StandardScaler(), ["total_sessions", "avg_session_length_minutes"]),  # Scale numerics only
    (OneHotEncoder(handle_unknown='ignore', sparse_output=False), ["experience", "subscribe"]),  # Encode categoricals
    remainder="drop"  # Drop target column and other non-features
)

# Now rebuild pipeline
tp_pipeline = make_pipeline(
    tp_preprocessor,
    KNeighborsClassifier()
)
tp_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('standardscaler',
                                                  StandardScaler(),
                                                  ['total_sessions',
                                                   'avg_session_length_minutes']),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['experience',
                                                   'subscribe'])])),
                ('kneighborsclassifier', KNeighborsClassifier())])

In [20]:
tp_preprocessor = make_column_transformer(
                        (StandardScaler(), ["total_sessions", "avg_session_length_minutes", "total_session_time_minutes"]),
                        remainder = "passthrough",
                        verbose_feature_names_out = False
)

tp_preprocessor.fit(tp_train)
tp_scaled = tp_preprocessor.transform(tp_train)
tp_scaled.head()

,total_sessions,avg_session_length_minutes,total_session_time_minutes,experience,subscribe,hashedEmail,name,gender,age,high_data_player
93,-0.267863,-0.464968,-0.270330,Veteran,True,5a340c0e3d1aa3e579efc625bd3e5bca7fc25f7115b68e...,Zoe,Male,20,False
39,-0.289246,-0.617260,-0.278250,Amateur,True,3caa832978e0596779f4ee7c686c4592fb6de893925025...,Thatcher,Male,22,False
1412,-0.289246,0.059596,-0.270708,Regular,True,d43af3bed5e9f1f31077233697c18f3f988a217bd0376a...,Xia,Female,20,False
846,-0.289246,-0.143461,-0.272970,Veteran,True,e44041459da2102dc20147ed6f0db4753547be66fc4dde...,Gianna,Male,18,False
243,-0.160947,0.262653,-0.182459,Regular,True,f2826fb8dbce4d450348f99cb27ade184b713998d96797...,Zane,Male,10,True


In [21]:


feature_cols = ['total_sessions', 'avg_session_length_minutes', 'experience', 'subscribe']
X = tp_train[feature_cols]
y = tp_train['high_data_player']

# Preprocessor 
tp_preprocessor = make_column_transformer(
    (StandardScaler(), ["total_sessions", "avg_session_length_minutes"]),
    (OneHotEncoder(handle_unknown='ignore', sparse_output=False), ["experience", "subscribe"]),
    remainder="drop"
)


tp_pipeline = make_pipeline(
    tp_preprocessor,
    KNeighborsClassifier(n_neighbors=5, metric='euclidean')  # k=5 common choice
)

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=123)

accuracy_scores = cross_val_score(tp_pipeline, X, y, cv=kf, scoring='accuracy')

print("KNN Accuracy scores:", accuracy_scores)
print(f"Mean Accuracy: {accuracy_scores.mean():.3f} (+/- {accuracy_scores.std() * 2:.3f})")


KNN Accuracy scores: [0.68421053 0.84210526 0.89473684 0.94444444 0.94444444]
Mean Accuracy: 0.862 (+/- 0.193)


In [23]:
# Test different k values
k_values = range(1, 11)
cv_scores = []

for k in k_values:
    knn = make_pipeline(
        tp_preprocessor,
        KNeighborsClassifier(n_neighbors=k)
    )
    scores = cross_val_score(knn, X, y, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f"Best k: {best_k} (F1: {max(cv_scores):.3f})")


Best k: 2 (F1: 0.871)


In [24]:
param_grid = {
    "kneighborsclassifier__n_neighbors": range(1, 11, 1),
}
tp_pipe = make_pipeline(tp_preprocessor, KNeighborsClassifier())

knn_tune_grid = GridSearchCV(
    tp_pipe, param_grid, cv = 5,
)
knn_tune_grid

knn_model_grid = knn_tune_grid.fit(tp_train[['total_sessions', 'avg_session_length_minutes', 'experience', 'subscribe']], tp_train["high_data_player"])
accuracies_grid = pd.DataFrame(knn_model_grid.cv_results_)

accuracies_grid

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kneighborsclassifier__n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.006721,0.000354,0.006001,0.000393,1,{'kneighborsclassifier__n_neighbors': 1},0.894737,0.894737,0.789474,0.833333,0.833333,0.849123,0.040541,8
1,0.006237,0.000080,0.006481,0.001843,2,{'kneighborsclassifier__n_neighbors': 2},0.947368,0.789474,0.894737,0.888889,0.833333,0.870760,0.054370,1
2,0.006183,0.000120,0.005455,0.000033,3,{'kneighborsclassifier__n_neighbors': 3},1.000000,0.789474,0.842105,0.833333,0.833333,0.859649,0.072548,6
3,0.006079,0.000013,0.005475,0.000035,4,{'kneighborsclassifier__n_neighbors': 4},0.947368,0.789474,0.947368,0.833333,0.833333,0.870175,0.065031,2
4,0.006088,0.000054,0.005432,0.000016,5,{'kneighborsclassifier__n_neighbors': 5},0.947368,0.789474,1.000000,0.777778,0.833333,0.869591,0.088565,5
5,0.006155,0.000086,0.005419,0.000021,6,{'kneighborsclassifier__n_neighbors': 6},0.947368,0.789474,0.947368,0.777778,0.888889,0.870175,0.073916,3
6,0.006038,0.000007,0.005421,0.000019,7,{'kneighborsclassifier__n_neighbors': 7},0.947368,0.789474,0.842105,0.777778,0.888889,0.849123,0.063136,8
7,0.006068,0.000068,0.005420,0.000014,8,{'kneighborsclassifier__n_neighbors': 8},0.947368,0.789474,0.947368,0.722222,0.833333,0.847953,0.088554,10
8,0.006292,0.000364,0.005479,0.000091,9,{'kneighborsclassifier__n_neighbors': 9},0.947368,0.789474,0.947368,0.722222,0.888889,0.859064,0.089502,7
9,0.006075,0.000050,0.005485,0.000116,10,{'kneighborsclassifier__n_neighbors': 10},0.947368,0.789474,0.947368,0.777778,0.888889,0.870175,0.073916,3


In [25]:
accuracy_versus_k_grid = alt.Chart(accuracies_grid).mark_line(point = True).encode(
    x = alt.X("param_kneighborsclassifier__n_neighbors")
    .title("N Neighbors")
    .scale(zero = False),
    y = alt.Y("mean_test_score")
    .title("Mean test score")
    .scale(zero = False)
)
accuracy_versus_k_grid

alt.Chart(...)

In [26]:
# Use best k for final model
tp_pipeline = make_pipeline(
    tp_preprocessor,
    KNeighborsClassifier(n_neighbors=best_k)
)

tp_pipeline.fit(X, y)
X_test = tp_test[feature_cols]
y_test = tp_test['high_data_player']

test_accuracy = tp_pipeline.score(X_test, y_test)
print(f"Test Accuracy with k={best_k}: {test_accuracy:.3f}")


Test Accuracy with k=2: 0.906


## Discussion

- summarize what you found
- discuss whether this is what you expected to find?
- discuss what impact could such findings have?
- discuss what future questions could this lead to?

## References